# AutoEIT Test I: Transcription submission
This notebook sequentially executes the baseline ASR transcription pipeline targeting exact learner production. It enforces a strict "draft" mentality, requiring offline manual review to fix ASR autocorrects and align the 30 target items accurately.

## 1. Objective & Setup
The Spanish Elicited Imitation Task (EIT) uses a meaning-based scoring rubric (0-4). Learner transcripts must exactly capture their production, including false starts, `[no response]` blocks, and grammatical deviances. Traditional ASRs naturally attempt to repair broken grammar, destroying the validity of the rubric.

**Execution Flow:**
1. Install `faster-whisper` for Colab T4 GPU execution.
2. Run `src/01_run_whisper.py` to baseline the audio tracking.
3. Run `src/02_prepare_audit_csv.py` to bridge the draft ASR to the mandatory audit step.
4. Suspend execution for the offline Audacity manual audit.
5. Run `src/03_populate_excel.py` to safely inject the 30 verified targets back into the template workbook.

## 2. Environment Configuration
*(You should change the Colab Runtime to GPU before executing this cell)*

In [ ]:
!pip install faster-whisper pandas openpyxl pyyaml
import os
# Verify config file maps to our exact files and offsets
!cat config/test1_metadata.yaml

## 3. Baseline ASR Generation
Running the raw generation locally or on colab GPU. We fast forward past the English instructions based on our yaml config.

In [ ]:
!python src/01_run_whisper.py

## 4. Audit CSV Preparation
This merges the target stimuli against the draft chronological texts.

In [ ]:
!python src/02_prepare_audit_csv.py

## 5. 🛑 STOP: Manual Auditing Protocol (MANDATORY)
You must now download the generated `outputs/review_csv/[id]_audit.csv` files and open them locally.

**Audacity Checklist:**
- Open the audio file.
- Wait for the sharp tone.
- Only transcribe exactly what the learner says *after* the tone.
- Ensure you transcribe "XXX" for unintelligible and "la-" for false starts.
- Do NOT normalize the learner's Spanish. If they said `la carro`, type `la carro`.
- Map exactly 30 unique items (1-30). Delete extra drift rows. If they completely missed an item, write `[no response]` in `FINAL_AUDIT_TRANSCRIPTION`.

*Once completed, upload the audited CSVs back to `outputs/review_csv/` before running the final workbook injection.*

## 6. Excel Write-Back Validation
Injecting the finalized sequences into Column C.

In [ ]:
!python src/03_populate_excel.py

## 7. Reflections & GSoC Proposal Strategy
Given the meaning-based 0-4 constraints established by the experimental protocol, pure ASR architectures fail because they behave like aggressive spelling-correctors. The long-term GSoC strategy must implement a constraint-aware alignment layer (Phase 3 of the proposal) that performs target-aware decoding against the known Column B stimulus string, optimizing for phonetic proximity rather than semantic fluency.